# 05. Prediction Stability & Measurement Sensitivity Analysis (E8)
**Role 2: Trust Engine Engineer**  
**Handbook Reference:** Section 1.2, Section 7.3 (Stage 4 Stability & Stage 5 Verdict), Section 7.6 (E8)

---

### Methodological Status Notice: PROVISIONAL BASELINE EVALUATION
> **IMPORTANT:** This notebook evaluates prediction stability and per-feature sensitivity on the current baseline pipeline (`StandardScaler + RandomForestClassifier`). Role 1 (Abhishek) has not yet merged the final calibrated model (`CalibratedClassifierCV`).  
> **All numerical values in this notebook are provisional.** Once Role 1 delivers the calibrated model, this exact workflow will be rerun to finalize production stability metrics and thresholds.

### Strict Methodological Safeguards:
1. **Zero Data Leakage:** Evaluated exclusively on the 330 held-out validation samples (`data/splits/val_idx.npy`). The test set (`data/splits/test_idx.npy`) remains strictly untouched.
2. **Approved Physical Noise Model:** Utilizes `backend.noise.apply_noise` with approved 1-sigma standard deviations and physical boundary clipping (`backend/models/noise_model.json`).
3. **Deterministic Reproducibility:** Fixed random seeds (`seed=42`) ensure identical results across runs.
4. **Vectorized Inference:** Efficient single-call NumPy batch processing ($M=200$ samples across $N=330$ inputs = 66,000 predictions).


In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from backend.noise import apply_noise, load_noise_config, FEATURE_NAMES, PHYSICAL_BOUNDS

# Matplotlib aesthetic defaults
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

SEED = 42
N_SAMPLES = 200

print(f"Project root: {PROJECT_ROOT}")
print(f"Canonical features: {list(FEATURE_NAMES)}")
print(f"Default perturbation samples (M): {N_SAMPLES}")
print(f"Global seed: {SEED}")


Project root: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine
Canonical features: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
Default perturbation samples (M): 200
Global seed: 42


In [2]:
# 1. Load dataset and split indices
data_path = os.path.join(PROJECT_ROOT, "data", "Crop_recommendation.csv")
splits_dir = os.path.join(PROJECT_ROOT, "data", "splits")

df = pd.read_csv(data_path)
train_idx = np.load(os.path.join(splits_dir, "train_idx.npy"))
val_idx = np.load(os.path.join(splits_dir, "val_idx.npy"))
test_idx = np.load(os.path.join(splits_dir, "test_idx.npy"))

# Verify zero overlap across splits
assert len(set(train_idx) & set(val_idx)) == 0, "Train and Val overlap!"
assert len(set(val_idx) & set(test_idx)) == 0, "Val and Test overlap!"
assert len(set(train_idx) & set(test_idx)) == 0, "Train and Test overlap!"

# Extract validation set as DataFrame to retain feature names
X_val_df = df.iloc[val_idx][list(FEATURE_NAMES)].copy()
X_val = X_val_df.values
y_val = np.array(df.iloc[val_idx]["label"].values, dtype=str)

# 2. Load model artifacts and noise config
models_dir = os.path.join(PROJECT_ROOT, "backend", "models")
model_path = os.path.join(models_dir, "crop_pipeline.joblib")
classes_path = os.path.join(models_dir, "classes.json")
noise_config_path = os.path.join(models_dir, "noise_model.json")

model = joblib.load(model_path)
with open(classes_path, "r", encoding="utf-8") as f:
    classes = json.load(f)
noise_config = load_noise_config(noise_config_path)

# 3. Clean Validation Performance
base_preds = model.predict(X_val_df)
clean_acc = np.mean(base_preds == y_val)

print(f"Dataset Total Rows:    {len(df)}")
print(f"Validation Set Rows:   {len(X_val)} (22 crops x 15 samples each)")
print(f"Model Pipeline:        {type(model).__name__} ({type(model.named_steps['classifier']).__name__})")
print(f"Clean Val Accuracy:    {clean_acc * 100:.2f}% ({np.sum(base_preds == y_val)}/{len(y_val)})")
print(f"Noise Profile:         '{noise_config['default_profile']}' (1-sigma Gaussian + boundary clipping)")


C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\varsh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Dataset Total Rows:    2200
Validation Set Rows:   330 (22 crops x 15 samples each)
Model Pipeline:        Pipeline (RandomForestClassifier)
Clean Val Accuracy:    99.09% (327/330)
Noise Profile:         'primary_1sigma' (1-sigma Gaussian + boundary clipping)


---
## Part 1: Batch Vectorized Stability Analysis (200 Noisy Copies per Input)

### Mathematical Definition
For a given input vector $x \in \mathbb{R}^7$, stability measures the fraction of $M=200$ independent physical noise perturbations for which the model's prediction remains identical to its base prediction:

$$\\text{stability}(x) = \\frac{1}{M} \\sum_{m=1}^M \\mathbb{I}\\left(\\hat{y}(x + \\epsilon_m) == \\hat{y}(x)\\right), \\quad \\epsilon_m \\sim \\mathcal{N}(0, \\sigma^2(x))$$

$$\\text{flip\\_rate}(x) = 1.0 - \\text{stability}(x)$$

- $\\text{stability} = 1.0$ (100%): Robust prediction safely situated well within the crop's decision envelope.
- $\\text{stability} < 0.80$: Fragile prediction near a decision boundary; minor realistic sensor variation or seasonal microclimate drift can flip the recommendation.


In [3]:
# 1. Generate 200 noisy copies for each of the 330 validation inputs in a single vectorized batch
noisy_3d, clip_stats = apply_noise(
    X_val,
    n_samples=N_SAMPLES,
    seed=SEED,
    profile="primary_1sigma",
    return_clip_stats=True
)

print(f"Generated Noisy Tensor Shape: {noisy_3d.shape} (samples x inputs x features)")
print(f"Total Perturbed Elements:     {clip_stats['total_elements']:,}")
print(f"Physical Boundary Clip Rate:  {clip_stats['clip_rate'] * 100:.2f}% ({clip_stats['total_clipped']:,} elements)")
print(f"Per-Feature Clip Counts:      {clip_stats['per_feature_clipped']}")

# 2. Vectorized batch inference across all 66,000 perturbed inputs
noisy_2d = noisy_3d.reshape(-1, len(FEATURE_NAMES))
noisy_df = pd.DataFrame(noisy_2d, columns=list(FEATURE_NAMES))
all_noisy_preds = model.predict(noisy_df).reshape(N_SAMPLES, len(X_val))

# 3. Compute per-input stability and flip rates
stability_scores = np.mean(all_noisy_preds == base_preds[None, :], axis=0)
flip_rates = 1.0 - stability_scores

# 4. Summary Statistics
print("\n" + "=" * 60)
print("PROVISIONAL VALIDATION STABILITY SUMMARY (M=200)")
print("=" * 60)
print(f"Mean Stability:             {np.mean(stability_scores) * 100:.2f}%")
print(f"Median Stability:           {np.median(stability_scores) * 100:.2f}%")
print(f"Std Deviation:              {np.std(stability_scores) * 100:.2f}%")
print(f"10th Percentile:            {np.percentile(stability_scores, 10) * 100:.2f}%")
print(f"Min Stability:              {np.min(stability_scores) * 100:.2f}%")
print(f"Mean Flip Rate:             {np.mean(flip_rates) * 100:.2f}%")
print(f"Fully Stable (100%):        {np.sum(stability_scores == 1.0)} / {len(stability_scores)} ({np.mean(stability_scores == 1.0) * 100:.2f}%)")
print(f"Fragile Inputs (<80%):      {np.sum(stability_scores < 0.80)} / {len(stability_scores)} ({np.mean(stability_scores < 0.80) * 100:.2f}%)")
print(f"Severely Fragile (<50%):    {np.sum(stability_scores < 0.50)} / {len(stability_scores)} ({np.mean(stability_scores < 0.50) * 100:.2f}%)")


Generated Noisy Tensor Shape: (200, 330, 7) (samples x inputs x features)
Total Perturbed Elements:     462,000
Physical Boundary Clip Rate:  0.57% (2,654 elements)
Per-Feature Clip Counts:      {'N': 1048, 'P': 5, 'K': 2, 'temperature': 0, 'humidity': 1599, 'ph': 0, 'rainfall': 0}



PROVISIONAL VALIDATION STABILITY SUMMARY (M=200)
Mean Stability:             91.95%
Median Stability:           97.50%
Std Deviation:              12.07%
10th Percentile:            75.00%
Min Stability:              31.50%
Mean Flip Rate:             8.05%
Fully Stable (100%):        91 / 330 (27.58%)
Fragile Inputs (<80%):      46 / 330 (13.94%)
Severely Fragile (<50%):    5 / 330 (1.52%)


---
## Part 2: The Stability-Accuracy Connection (Empirical Proof of E8)

A critical hypothesis in the Crop Trust Engine design (Handbook Section 1.2 & 7.3) is:
> *"Show that low-stability answers are wrong more often than high-stability ones, using noisy test inputs."*

Here, we empirically test this hypothesis on the validation set by comparing the actual classification accuracy under noise for **high-stability inputs** ($\ge 0.80$) versus **fragile inputs** ($< 0.80$).


In [4]:
# Compute noisy classification accuracy against ground truth y_val
noisy_correct = (all_noisy_preds == y_val[None, :])
overall_noisy_acc = np.mean(noisy_correct)

high_stab_mask = stability_scores >= 0.80
low_stab_mask = stability_scores < 0.80

n_high = int(np.sum(high_stab_mask))
n_low = int(np.sum(low_stab_mask))

high_stab_noisy_acc = float(np.mean(noisy_correct[:, high_stab_mask]))
low_stab_noisy_acc = float(np.mean(noisy_correct[:, low_stab_mask]))

high_stab_err = 1.0 - high_stab_noisy_acc
low_stab_err = 1.0 - low_stab_noisy_acc
error_spike = low_stab_err / high_stab_err if high_stab_err > 0 else float("inf")

print("=" * 70)
print("ACCURACY UNDER PHYSICAL NOISE BY STABILITY SEGMENT")
print("=" * 70)
print(f"Clean Validation Accuracy:               {clean_acc * 100:.2f}%")
print(f"Overall Noisy Validation Accuracy:       {overall_noisy_acc * 100:.2f}% (66,000 evaluations)")
print(f"High-Stability Inputs (>=80%):          {high_stab_noisy_acc * 100:.2f}% accuracy ({n_high} inputs)")
print(f"Low-Stability Inputs (<80%):            {low_stab_noisy_acc * 100:.2f}% accuracy ({n_low} inputs)")
print(f"Error Rate Comparison:                   {high_stab_err * 100:.2f}% (High) vs. {low_stab_err * 100:.2f}% (Low)")
print(f"Relative Error Risk Spike:               {error_spike:.1f}x higher error rate in low-stability inputs")

# Visualization: Stability Distribution & Error Rate Comparison
os.makedirs(os.path.join(PROJECT_ROOT, "docs", "figures"), exist_ok=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Histogram & KDE of stability scores
ax1.hist(stability_scores * 100, bins=25, color="#1f77b4", edgecolor="#0a3d62", alpha=0.75, density=False)
ax1.axvline(80.0, color="#d62728", linestyle="--", linewidth=2, label="Fragility Threshold (80%)")
ax1.set_title("Distribution of Stability Scores (M=200)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Stability Score (%)", fontsize=11)
ax1.set_ylabel("Number of Inputs", fontsize=11)
ax1.legend(loc="upper left")
ax1.grid(True, linestyle=":", alpha=0.6)

# Plot 2: Accuracy & Error Rate under Noise
bars = ax2.bar(
    ["High-Stability\n(>=80%, n=284)", "Low-Stability\n(<80%, n=46)"],
    [high_stab_noisy_acc * 100, low_stab_noisy_acc * 100],
    color=["#2ca02c", "#d62728"],
    edgecolor="#333333",
    width=0.5
)
ax2.set_ylabel("Noisy Accuracy (%)", fontsize=11)
ax2.set_title("Accuracy Under Noise by Stability Segment", fontsize=12, fontweight="bold")
ax2.set_ylim(0, 110)
ax2.grid(True, axis="y", linestyle=":", alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, yval + 2, f"{yval:.2f}%", ha="center", va="bottom", fontweight="bold")

fig_path_dist = os.path.join(PROJECT_ROOT, "docs", "figures", "stability_distribution.png")
plt.tight_layout()
plt.savefig(fig_path_dist, dpi=200)
plt.close()
print(f"\nSaved figure to: {fig_path_dist}")


ACCURACY UNDER PHYSICAL NOISE BY STABILITY SEGMENT
Clean Validation Accuracy:               99.09%
Overall Noisy Validation Accuracy:       91.92% (66,000 evaluations)
High-Stability Inputs (>=80%):          96.07% accuracy (284 inputs)
Low-Stability Inputs (<80%):            66.32% accuracy (46 inputs)
Error Rate Comparison:                   3.93% (High) vs. 33.68% (Low)
Relative Error Risk Spike:               8.6x higher error rate in low-stability inputs



Saved figure to: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\stability_distribution.png


---
## Part 3: Per-Crop Stability & Flip-Rate Breakdown

Different crops occupy different geometric clusters in $\mathbb{R}^7$. Some crops (like orange or chickpea) occupy isolated physiological envelopes and exhibit near-100% stability. Other crops (like blackgram, jute, and rice) border neighbor clusters, leading to boundary sensitivity under realistic noise.


In [5]:
crop_records = []
for crop in sorted(np.unique(y_val)):
    mask = (y_val == crop)
    c_stabs = stability_scores[mask]
    c_flips = flip_rates[mask]
    crop_records.append({
        "crop": crop,
        "n_samples": int(np.sum(mask)),
        "mean_stability": float(np.mean(c_stabs)),
        "median_stability": float(np.median(c_stabs)),
        "min_stability": float(np.min(c_stabs)),
        "mean_flip_rate": float(np.mean(c_flips)),
        "fragile_count": int(np.sum(c_stabs < 0.80)),
        "fragile_pct": float(np.mean(c_stabs < 0.80) * 100)
    })

crop_df = pd.DataFrame(crop_records).sort_values("mean_stability", ascending=True).reset_index(drop=True)

print("=" * 90)
print("PER-CROP STABILITY BREAKDOWN (SORTED BY MEAN STABILITY)")
print("=" * 90)
print(crop_df.to_string(index=False, formatters={
    "mean_stability": lambda x: f"{x*100:6.2f}%",
    "median_stability": lambda x: f"{x*100:6.2f}%",
    "min_stability": lambda x: f"{x*100:6.2f}%",
    "mean_flip_rate": lambda x: f"{x*100:6.2f}%",
    "fragile_pct": lambda x: f"{x:5.1f}%"
}))

# Plot: Horizontal bar chart ranking crops by stability
fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(crop_df))
colors = ["#d62728" if x < 0.85 else "#ff7f0e" if x < 0.95 else "#2ca02c" for x in crop_df["mean_stability"]]

bars = ax.barh(y_pos, crop_df["mean_stability"] * 100, color=colors, edgecolor="#333333", height=0.65)
ax.axvline(80.0, color="#d62728", linestyle="--", linewidth=1.5, label="80% Stability Threshold")
ax.set_yticks(y_pos)
ax.set_yticklabels(crop_df["crop"], fontsize=10)
ax.set_xlabel("Mean Prediction Stability (%)", fontsize=11)
ax.set_title("Per-Crop Prediction Stability Under 1-Sigma Noise (22 Crops)", fontsize=12, fontweight="bold")
ax.set_xlim(50, 105)
ax.legend(loc="lower right")
ax.grid(True, axis="x", linestyle=":", alpha=0.6)

fig_path_crop = os.path.join(PROJECT_ROOT, "docs", "figures", "stability_per_crop.png")
plt.tight_layout()
plt.savefig(fig_path_crop, dpi=200)
plt.close()
print(f"\nSaved per-crop figure to: {fig_path_crop}")


PER-CROP STABILITY BREAKDOWN (SORTED BY MEAN STABILITY)
       crop  n_samples mean_stability median_stability min_stability mean_flip_rate  fragile_count fragile_pct
  blackgram         15         68.83%           73.00%        31.50%         31.17%             11       73.3%
       jute         15         69.40%           67.00%        46.00%         30.60%             11       73.3%
     lentil         15         81.50%           85.00%        50.50%         18.50%              6       40.0%
       rice         15         81.60%           86.00%        37.00%         18.40%              6       40.0%
     cotton         15         86.60%           86.50%        73.00%         13.40%              4       26.7%
      maize         15         88.50%           94.50%        50.50%         11.50%              3       20.0%
kidneybeans         15         91.13%           91.00%        76.50%          8.87%              1        6.7%
      apple         15         92.60%           92.50%  


Saved per-crop figure to: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\stability_per_crop.png


---
## Part 4: Controlled Per-Feature Sensitivity Curves

To understand **which individual environmental or soil features drive prediction flips**, we perturb each feature individually across a physical perturbation grid $\delta \in [-3.0\sigma, +3.0\sigma]$ while holding the other 6 features constant.

- **Relative features (N, P, K, rainfall):** Scaled proportional to the feature mean with additive floors.
- **Absolute features (temperature, humidity, pH):** Stepped in physical units ($1.5^\circ\text{C}, 5.0\%, 0.3\text{ pH}$).


In [6]:
# Define 1-sigma perturbation step for each feature
sigma_dict = {
    "N": max(0.10 * np.mean(X_val[:, 0]), 2.0),
    "P": max(0.10 * np.mean(X_val[:, 1]), 2.0),
    "K": max(0.10 * np.mean(X_val[:, 2]), 2.0),
    "temperature": 1.5,
    "humidity": 5.0,
    "ph": 0.3,
    "rainfall": max(0.10 * np.mean(X_val[:, 6]), 5.0),
}

steps = np.linspace(-3.0, 3.0, 13)
curves = {feat: [] for feat in FEATURE_NAMES}

for f_idx, feat in enumerate(FEATURE_NAMES):
    s = sigma_dict[feat]
    low_b, high_b = PHYSICAL_BOUNDS[feat]
    for step in steps:
        delta = step * s
        X_pert = X_val.copy()
        X_pert[:, f_idx] = np.clip(X_pert[:, f_idx] + delta, low_b, high_b)
        pert_df = pd.DataFrame(X_pert, columns=list(FEATURE_NAMES))
        preds = model.predict(pert_df)
        retention = np.mean(preds == base_preds)
        curves[feat].append(retention)

sensitivity_df = pd.DataFrame(curves, index=[f"{s:+.1f}s" for s in steps])

print("=" * 80)
print("FEATURE SENSITIVITY: PREDICTION RETENTION (%) AS FEATURE IS SHIFTED")
print("=" * 80)
print((sensitivity_df * 100).round(2).to_string())

# Plot: Multi-panel sensitivity curves
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.flatten()

colors_feat = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]

for idx, feat in enumerate(FEATURE_NAMES):
    ax = axes[idx]
    y_vals = np.array(curves[feat]) * 100
    ax.plot(steps, y_vals, marker="o", color=colors_feat[idx], linewidth=2, label=f"1s = {sigma_dict[feat]:.1f}")
    ax.axhline(100.0, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(f"{feat}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Perturbation Magnitude (in sigmas)", fontsize=10)
    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(75, 102)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="lower left", fontsize=9)

axes[0].set_ylabel("Prediction Retention (%)", fontsize=11)
axes[4].set_ylabel("Prediction Retention (%)", fontsize=11)

# Hide 8th unused subplot
axes[7].axis("off")

plt.suptitle("Controlled Per-Feature Sensitivity Curves (Clean Val Set, n=330)", fontsize=14, fontweight="bold")
fig_path_sens = os.path.join(PROJECT_ROOT, "docs", "figures", "sensitivity_curves.png")
plt.tight_layout()
plt.savefig(fig_path_sens, dpi=200)
plt.close()
print(f"\nSaved sensitivity curves to: {fig_path_sens}")


FEATURE SENSITIVITY: PREDICTION RETENTION (%) AS FEATURE IS SHIFTED
            N       P       K  temperature  humidity      ph  rainfall
-3.0s   98.18   94.85   95.15        97.88     83.03   97.27     88.18
-2.5s   98.18   95.45   97.58        98.18     86.06   97.58     90.00
-2.0s   98.79   96.36   99.39        99.09     88.79   97.58     92.12
-1.5s   99.09   98.48  100.00        99.39     93.94   97.88     93.94
-1.0s   99.70   99.09  100.00        99.39     96.36   98.18     96.97
-0.5s   99.70   99.39  100.00       100.00     99.39   99.09     99.09
+0.0s  100.00  100.00  100.00       100.00    100.00  100.00    100.00
+0.5s   99.09   99.39  100.00       100.00     99.39  100.00    100.00
+1.0s   98.48   99.39  100.00        99.70     98.79  100.00     96.36
+1.5s   98.18   99.39   99.70        99.39     93.94  100.00     93.33
+2.0s   98.18   99.39   99.70        99.09     90.61  100.00     89.09
+2.5s   97.88   99.39   97.88        99.09     86.97  100.00     85.15
+3.0s   9


Saved sensitivity curves to: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\sensitivity_curves.png


---
## Part 5: Real Fragile Validation Inputs & Decision Margin Traversal

Here we examine the specific validation rows where stability is lowest. These examples directly demonstrate the necessity of the Trust Engine: **on paper, the clean model achieved 99% accuracy; in reality, small sensor noise flips these recommendations**.


In [7]:
fragile_sort_idx = np.argsort(stability_scores)[:5]

print("=" * 85)
print("DETAILED INSPECTION OF TOP 5 MOST FRAGILE VALIDATION INPUTS")
print("=" * 85)

for rank, idx in enumerate(fragile_sort_idx, start=1):
    val_row_id = val_idx[idx]
    true_label = y_val[idx]
    base_pred = base_preds[idx]
    stab = stability_scores[idx]
    noisy_dist = Counter(all_noisy_preds[:, idx]).most_common(4)
    
    print(f"Rank {rank} | Val Index: {val_row_id} | True Crop: {true_label} | Base Prediction: {base_pred}")
    print(f"  Stability Score: {stab * 100:.1f}% (Flip Rate: {(1.0 - stab) * 100:.1f}%)")
    print("  Input Features:")
    for feat in FEATURE_NAMES:
        print(f"    - {feat:12s}: {X_val[idx, FEATURE_NAMES.index(feat)]:.2f}")
    print("  Prediction Distribution Under 200 Noisy Samples:")
    for crop_name, cnt in noisy_dist:
        pct = (cnt / N_SAMPLES) * 100
        print(f"    * {crop_name:12s}: {cnt:3d} / {N_SAMPLES} ({pct:5.1f}%)")
    print("-" * 85)


DETAILED INSPECTION OF TOP 5 MOST FRAGILE VALIDATION INPUTS
Rank 1 | Val Index: 754 | True Crop: blackgram | Base Prediction: blackgram
  Stability Score: 31.5% (Flip Rate: 68.5%)
  Input Features:
    - N           : 22.00
    - P           : 55.00
    - K           : 20.00
    - temperature : 33.95
    - humidity    : 69.96
    - ph          : 7.42
    - rainfall    : 61.16
  Prediction Distribution Under 200 Noisy Samples:
    * mothbeans   :  74 / 200 ( 37.0%)
    * blackgram   :  63 / 200 ( 31.5%)
    * mungbean    :  34 / 200 ( 17.0%)
    * lentil      :  28 / 200 ( 14.0%)
-------------------------------------------------------------------------------------
Rank 2 | Val Index: 83 | True Crop: rice | Base Prediction: jute
  Stability Score: 37.0% (Flip Rate: 63.0%)
  Input Features:
    - N           : 67.00
    - P           : 43.00
    - K           : 39.00
    - temperature : 26.04
    - humidity    : 84.97
    - ph          : 6.00
    - rainfall    : 186.75
  Prediction Distri

---
## Part 6: Noise Profile Monotonicity: Primary 1-Sigma vs. Mild Half-Sigma

The approved noise model specification (`noise_model.json`) defines two profiles:
1. `primary_1sigma`: Full standard deviation (default).
2. `mild_half_sigma`: Halved perturbation ($0.5\sigma$).

Evaluating both profiles demonstrates monotonic consistency: as perturbation decreases, prediction stability increases toward 100%.


In [8]:
noisy_half_3d = apply_noise(
    X_val,
    n_samples=N_SAMPLES,
    seed=SEED,
    profile="mild_half_sigma"
)
all_preds_half = model.predict(pd.DataFrame(noisy_half_3d.reshape(-1, len(FEATURE_NAMES)), columns=list(FEATURE_NAMES))).reshape(N_SAMPLES, len(X_val))
stab_half = np.mean(all_preds_half == base_preds[None, :], axis=0)

print("=" * 70)
print("MONOTONICITY CHECK: PRIMARY 1-SIGMA VS. MILD HALF-SIGMA")
print("=" * 70)
print(f"Mild Half-Sigma Mean Stability:  {np.mean(stab_half) * 100:.2f}% (Median: {np.median(stab_half) * 100:.2f}%, Min: {np.min(stab_half) * 100:.2f}%)")
print(f"Primary 1-Sigma Mean Stability:  {np.mean(stability_scores) * 100:.2f}% (Median: {np.median(stability_scores) * 100:.2f}%, Min: {np.min(stability_scores) * 100:.2f}%)")
print(f"Half-Sigma Fragile Inputs (<80%): {np.sum(stab_half < 0.80)} / {len(X_val)}")
print(f"Full-Sigma Fragile Inputs (<80%): {np.sum(stability_scores < 0.80)} / {len(X_val)}")

assert np.mean(stab_half) >= np.mean(stability_scores), "Monotonicity violated!"
print("\nMonotonicity Check: PASSED.")


MONOTONICITY CHECK: PRIMARY 1-SIGMA VS. MILD HALF-SIGMA
Mild Half-Sigma Mean Stability:  97.65% (Median: 100.00%, Min: 48.50%)
Primary 1-Sigma Mean Stability:  91.95% (Median: 97.50%, Min: 31.50%)
Half-Sigma Fragile Inputs (<80%): 12 / 330
Full-Sigma Fragile Inputs (<80%): 46 / 330

Monotonicity Check: PASSED.


---
## Part 7: Synthesis & Downstream Trust Engine Handoff

### Key Findings Summary:
1. **Average Flip Rate (E8):** Across the validation set, the baseline pipeline exhibits a mean stability of **91.95%** and a mean flip rate of **8.05%** under approved 1-sigma noise.
2. **Empirical Accuracy Link:** When subjected to physical noise, inputs with high stability ($\ge 0.80$) maintain **96.07%** accuracy, whereas fragile inputs ($< 0.80$) collapse to **66.32%** accuracy (**8.6x relative error risk**).
3. **Fragile Crops:** `blackgram` (68.8% mean stability), `jute` (69.4%), `lentil` (81.5%), and `rice` (81.6%) exhibit the highest boundary sensitivity.
4. **Primary Sensitive Features:** `humidity` (82.4% retention at $+3\sigma$) and `rainfall` (82.7% retention at $+3\sigma$) produce the fastest prediction flips.
5. **Next Steps for Calibrated Model:**
   - When Role 1 delivers `backend/models/crop_pipeline.joblib` (`CalibratedClassifierCV`), re-run this notebook to update metrics.
   - Integrate `assess_stability()` into `backend/trust_engine.py` (Stage 4).
   - Use the 80% stability cutoff in Stage 5 risk profiles to route fragile inputs to `"caution"`.


In [9]:
# Export provisional stability metrics summary
provisional_summary = {
    "experiment": "E8_stability_analysis",
    "status": "provisional_baseline",
    "model_type": "baseline_random_forest_uncalibrated",
    "evaluation_split": "validation",
    "n_val_samples": len(X_val),
    "n_perturbations_per_sample": N_SAMPLES,
    "noise_profile": "primary_1sigma",
    "overall_metrics": {
        "clean_validation_accuracy": float(clean_acc),
        "mean_stability": float(np.mean(stability_scores)),
        "median_stability": float(np.median(stability_scores)),
        "min_stability": float(np.min(stability_scores)),
        "std_stability": float(np.std(stability_scores)),
        "mean_flip_rate": float(np.mean(flip_rates)),
        "fully_stable_count": int(np.sum(stability_scores == 1.0)),
        "fragile_count_below_80": int(np.sum(stability_scores < 0.80)),
        "high_stability_noisy_accuracy": high_stab_noisy_acc,
        "low_stability_noisy_accuracy": low_stab_noisy_acc,
        "relative_error_spike": float(error_spike)
    },
    "most_fragile_crops": crop_df.head(4)[["crop", "mean_stability", "mean_flip_rate"]].to_dict(orient="records"),
    "most_stable_crops": crop_df.tail(4)[["crop", "mean_stability", "mean_flip_rate"]].to_dict(orient="records"),
    "figures": {
        "stability_distribution": "docs/figures/stability_distribution.png",
        "stability_per_crop": "docs/figures/stability_per_crop.png",
        "sensitivity_curves": "docs/figures/sensitivity_curves.png"
    }
}

summary_path = os.path.join(PROJECT_ROOT, "docs", "figures", "stability_provisional_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(provisional_summary, f, indent=2)

print(f"Exported provisional stability summary to: {summary_path}")
print("=" * 80)
print("TASK 4 COMPLETE: Notebook 05 executed and figures exported.")
print("=" * 80)


Exported provisional stability summary to: C:\Users\varsh\OneDrive\Desktop\crop-trust-engine\docs\figures\stability_provisional_summary.json
TASK 4 COMPLETE: Notebook 05 executed and figures exported.
